# Colab 실행 안내

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/glasslego/2026-ml-study/blob/main/notebooks/study_01_pytorch_core/02_training_pipeline.ipynb)

이 노트북은 **PyTorch 학습 루프의 부품들을 분해해서** 이해하는 데 집중한다.
01 노트북에서 본 autograd 위에 얹어진 `nn.Module`, `Optimizer`, `Dataset`/`DataLoader`,
`device` / `dtype` / `checkpoint` 가 각각 무엇을 책임지는지 짚는다.

흐름:

1. `nn.Module` 의 본질 — parameters / buffers / state_dict
2. Optimizer 의 동작 — `param_groups`, `step()`, `zero_grad()`
3. Dataset 과 DataLoader
4. 학습 루프 정석 패턴
5. eval 모드와 `torch.no_grad`
6. device 관리 (cpu / cuda / mps)
7. dtype 과 혼합정밀도 (autocast)
8. 체크포인트 저장/복원
9. 흔한 함정 (Common Pitfalls)

In [ ]:
# 공통 실행 환경 준비 (Colab/Local 통합)
# 이 셀은 Colab과 로컬 Jupyter 사이의 환경 차이를 흡수한다.
# DLFS_code 의 prepare_notebook_environment 는 lincoln 폴더에 의존하므로
# study_* 노트북에서는 더 단순한 부트스트랩을 직접 둔다.
from pathlib import Path
import subprocess
import sys
import os

REPO_URL = 'https://github.com/glasslego/2026-ml-study.git'
REPO_NAME = '2026-ml-study'
NOTEBOOK_SUBDIR = 'notebooks/study_01_pytorch_core'

if 'google.colab' in sys.modules:
    clone_root = Path('/content') / REPO_NAME
    if not clone_root.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(clone_root)], check=True)
    target = clone_root / NOTEBOOK_SUBDIR
else:
    target = None
    for c in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (c / NOTEBOOK_SUBDIR).exists():
            target = c / NOTEBOOK_SUBDIR
            break
    if target is None:
        raise FileNotFoundError(f'{NOTEBOOK_SUBDIR} 를 찾지 못했습니다.')

os.chdir(target)
print(f'작업 디렉토리: {target}')

# Colab 에는 PyTorch 가 기본 설치되어 있지만 로컬에 없을 수도 있어 안전하게 보강.
try:
    import torch  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch'], check=True)
    import torch  # noqa: F401

print(f'torch version: {torch.__version__}')

In [ ]:
# 핵심 import
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset

SEED = 20260427
np.random.seed(SEED)
torch.manual_seed(SEED)

# 1. `nn.Module` 의 본질 — parameters / buffers / state_dict

`nn.Module` 안에는 두 종류의 "저장값" 이 있다.

- **parameter**: `nn.Parameter` 로 선언된 텐서. **학습된다 (`.grad` 받고 optimizer 가 갱신).**
  - `model.parameters()` / `model.named_parameters()` 로 순회.
- **buffer**: `self.register_buffer(name, tensor)` 로 등록. **학습 안 되지만 state 에는 포함.**
  - 대표 예: `BatchNorm` 의 `running_mean`, `running_var`.

**`state_dict()`** 는 위 두 가지를 모두 묶어 dict 로 돌려준다 → 체크포인트 저장의 단위.

In [ ]:
# 작은 모델로 parameter / buffer / state_dict 비교
class TinyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(3, 2)         # weight, bias 가 parameter 로 자동 등록
        self.bn = nn.BatchNorm1d(2)       # running_mean, running_var 는 buffer
        # 직접 등록한 buffer (학습 안 되지만 state 에는 들어감)
        self.register_buffer('step_count', torch.zeros(1, dtype=torch.long))

    def forward(self, x):
        self.step_count += 1
        return self.bn(self.fc(x))

model = TinyModel()

print('--- named_parameters (학습 대상) ---')
for n, p in model.named_parameters():
    print(f'  {n:20s} shape={tuple(p.shape)} requires_grad={p.requires_grad}')

print('--- named_buffers (학습 대상 X, 저장 O) ---')
for n, b in model.named_buffers():
    print(f'  {n:20s} shape={tuple(b.shape)}')

print('--- state_dict 키 (parameter + buffer 합쳐짐) ---')
print(list(model.state_dict().keys()))

# 2. Optimizer 의 동작 — `param_groups`, `step()`, `zero_grad()`

옵티마이저는 내부적으로 **`param_groups`** 라는 리스트를 들고 있고, 각 그룹마다 다른 하이퍼파라미터(lr, weight_decay 등) 를 줄 수 있다.

- `optimizer.step()`: 각 파라미터의 `.grad` 를 보고 `param.data` 를 in-place 갱신.
- `optimizer.zero_grad(set_to_none=True)`: 다음 backward 전에 `.grad` 초기화. 기본값 `set_to_none=True` 가 메모리/속도 면에서 유리.

전형적인 활용은 **fine-tuning 시 backbone 은 작은 lr, head 는 큰 lr** 처럼 그룹별로 다른 lr 을 주는 패턴.

In [ ]:
# param_groups 로 backbone vs head 분리
torch.manual_seed(SEED)
backbone = nn.Sequential(nn.Linear(10, 20), nn.ReLU(), nn.Linear(20, 20))
head = nn.Linear(20, 1)

opt = optim.SGD([
    {'params': backbone.parameters(), 'lr': 1e-4},   # 작은 lr
    {'params': head.parameters(),     'lr': 1e-2},   # 큰 lr
], momentum=0.9)

for i, g in enumerate(opt.param_groups):
    n_params = sum(p.numel() for p in g['params'])
    print(f'group {i}: lr={g["lr"]:.0e}, momentum={g["momentum"]}, #params={n_params}')

# 3. Dataset 과 DataLoader

PyTorch 의 데이터 추상화는 두 단계.

- **`Dataset`**: `__len__`, `__getitem__(idx)` 만 구현하면 됨. 단일 샘플 로직.
- **`DataLoader`**: Dataset 을 받아 **배치화 + 셔플 + 병렬 로딩** 을 담당.

DataLoader 의 주요 인자:

| 인자 | 설명 |
|------|------|
| `batch_size` | 배치 크기 |
| `shuffle` | epoch 마다 인덱스 섞기 |
| `num_workers` | 멀티프로세스 로딩 (>0 이면 별도 워커) |
| `pin_memory` | GPU 전송 가속 (cuda 사용 시 True) |
| `collate_fn` | 샘플들을 배치로 묶는 함수. 가변 길이 시퀀스에 필수 |

In [ ]:
# 가짜 회귀 데이터로 커스텀 Dataset + DataLoader
class NoisyLineDataset(Dataset):
    """y = 3x + 2 + noise 형태의 가짜 1D 회귀 데이터."""
    def __init__(self, n: int = 200, seed: int = 0):
        rng = np.random.default_rng(seed)
        self.x = rng.uniform(-1, 1, size=(n, 1)).astype(np.float32)
        self.y = (3 * self.x + 2 + rng.normal(0, 0.1, size=(n, 1))).astype(np.float32)

    def __len__(self) -> int:
        return len(self.x)

    def __getitem__(self, idx: int):
        return torch.from_numpy(self.x[idx]), torch.from_numpy(self.y[idx])

ds = NoisyLineDataset(n=200, seed=SEED)
loader = DataLoader(ds, batch_size=32, shuffle=True, num_workers=0)

print(f'데이터셋 크기: {len(ds)}')
xb, yb = next(iter(loader))
print(f'한 배치 shape: x={tuple(xb.shape)}, y={tuple(yb.shape)}')

# 4. 학습 루프 정석 패턴

매 step:

1. `model.train()` — train 모드 (Dropout/BN 활성화)
2. `optimizer.zero_grad()` — 이전 grad 비우기
3. `pred = model(xb)` — forward
4. `loss = loss_fn(pred, yb)` — 손실
5. `loss.backward()` — 그래디언트 계산
6. `optimizer.step()` — 파라미터 갱신
7. `loss.item()` 누적 — **반드시 `.item()` 으로 스칼라화**

이 흐름을 함수로 캡슐화해 두면 학습/평가 코드가 깨끗해진다.

In [ ]:
def train_one_epoch(model, loader, loss_fn, optimizer, device='cpu'):
    model.train()
    total_loss, n = 0.0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad(set_to_none=True)
        pred = model(xb)
        loss = loss_fn(pred, yb)
        loss.backward()
        optimizer.step()
        # .item() 안 하면 그래프가 살아 있어 메모리 누수
        total_loss += loss.item() * xb.size(0)
        n += xb.size(0)
    return total_loss / n


@torch.no_grad()
def evaluate(model, loader, loss_fn, device='cpu'):
    model.eval()
    total_loss, n = 0.0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        pred = model(xb)
        loss = loss_fn(pred, yb)
        total_loss += loss.item() * xb.size(0)
        n += xb.size(0)
    return total_loss / n


# 짧은 학습 시연
torch.manual_seed(SEED)
model = nn.Sequential(nn.Linear(1, 16), nn.ReLU(), nn.Linear(16, 1))
opt = optim.SGD(model.parameters(), lr=0.05)
loss_fn = nn.MSELoss()

for ep in range(5):
    tr = train_one_epoch(model, loader, loss_fn, opt)
    ev = evaluate(model, loader, loss_fn)
    print(f'epoch {ep+1}: train={tr:.4f}  eval={ev:.4f}')

# 5. eval 모드와 `torch.no_grad`

두 가지가 **다른 일**을 한다는 점이 핵심이다.

- `model.eval()` / `model.train()`: **모듈의 동작 모드** 를 바꾼다.
  - `Dropout` 은 eval 에서 비활성 (출력 그대로 통과).
  - `BatchNorm` 은 eval 에서 mini-batch 통계 대신 누적된 `running_mean`/`running_var` 사용.
- `torch.no_grad()`: **그래프 추적을 끔.** 메모리/속도 절약. 학습/평가 모드와는 독립.

추론 시 보통 둘 다 쓴다 → `model.eval()` + `with torch.no_grad():`.

In [ ]:
# Dropout 으로 train() vs eval() 차이 확인
torch.manual_seed(SEED)
drop_model = nn.Sequential(nn.Linear(8, 8), nn.Dropout(p=0.5))

x = torch.ones(1, 8)

drop_model.train()
out_train_a = drop_model(x)
out_train_b = drop_model(x)
print('train mode (Dropout 활성):')
print('  실행 1:', out_train_a.detach().numpy().round(3))
print('  실행 2:', out_train_b.detach().numpy().round(3), ' <- 매번 다름')

drop_model.eval()
out_eval_a = drop_model(x)
out_eval_b = drop_model(x)
print('eval mode (Dropout 비활성):')
print('  실행 1:', out_eval_a.detach().numpy().round(3))
print('  실행 2:', out_eval_b.detach().numpy().round(3), ' <- 항상 같음')

# 6. device 관리 (cpu / cuda / mps)

규칙은 단순하다: **모델과 입력 텐서는 같은 device 위에 있어야 한다.**

- `cuda`: NVIDIA GPU
- `mps`: Apple Silicon GPU (M1/M2/M3)
- `cpu`: 항상 사용 가능

자동 감지 헬퍼를 두고 모든 텐서/모델을 한 곳으로 모은다.

In [ ]:
def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device('cuda')
    if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')


device = get_device()
print('선택된 device:', device)

# 모델/데이터 이동
torch.manual_seed(SEED)
m = nn.Linear(4, 2).to(device)
xb = torch.randn(3, 4).to(device)
yb = m(xb)
print('m.weight.device =', m.weight.device, ', xb.device =', xb.device, ', yb.device =', yb.device)

# 7. dtype 과 혼합정밀도 (autocast)

기본 dtype 은 `float32`. 메모리 절약과 속도 향상을 위해 일부 연산을 `float16` / `bfloat16` 으로 돌리는 것이 **혼합정밀도(mixed precision)** 다.

- `torch.autocast(device_type='cuda', dtype=torch.float16)` : 컨텍스트 안 연산을 자동으로 캐스팅.
- `torch.cuda.amp.GradScaler` : float16 의 underflow 를 막기 위해 loss 를 스케일.

CPU 에서는 GradScaler 가 동작하지 않으므로 아래 셀은 **GPU 환경에서만 실제로 돌린다.**
패턴만 익히는 용도.

In [ ]:
# 혼합정밀도 학습 패턴 (GPU 에서만 의미 있음)
# CPU 에서는 GradScaler 가 no-op 이거나 미지원이므로 코드 패턴만 보여 준다.

def train_step_amp(model, xb, yb, loss_fn, optimizer, scaler, device):
    """AMP 한 step. device='cuda' 에서만 실제 효과."""
    optimizer.zero_grad(set_to_none=True)
    with torch.autocast(device_type=device.type, dtype=torch.float16):
        pred = model(xb)
        loss = loss_fn(pred, yb)
    scaler.scale(loss).backward()    # underflow 방지로 loss 스케일 업
    scaler.step(optimizer)           # 내부에서 unscale 후 step
    scaler.update()                  # 다음 step 의 scale 갱신
    return loss.item()


print('AMP 패턴은 위 함수와 같다. cuda 가 있을 때만 아래 셋업이 의미 있다.')
if device.type == 'cuda':
    scaler = torch.cuda.amp.GradScaler()
    print('GradScaler 준비 완료 (cuda)')
else:
    print(f'현재 device={device.type} — AMP 효과 없음. GPU 에서 시도해 볼 것.')

# 8. 체크포인트 — 저장과 복원

저장해야 할 최소 단위:

- `model.state_dict()` — 모든 parameter + buffer
- `optimizer.state_dict()` — momentum 등 optimizer 내부 상태
- `epoch`, `best_loss` 등 학습 진행 메타데이터

**모델 객체 자체를 `torch.save(model, ...)`** 으로 저장하는 방식은 클래스 정의에 의존해 깨지기 쉽다.
가능하면 **state_dict 만 저장** 하는 패턴을 쓴다.

In [ ]:
# 체크포인트 저장 / 복원 시연
import tempfile

torch.manual_seed(SEED)
model = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 2))
opt = optim.Adam(model.parameters(), lr=1e-3)

# 한 step 만 돌려서 optimizer state 도 채워 둔다
xb, yb = torch.randn(5, 4), torch.randn(5, 2)
loss = nn.MSELoss()(model(xb), yb)
loss.backward()
opt.step()

ckpt = {
    'epoch': 7,
    'best_loss': 0.123,
    'model_state': model.state_dict(),
    'optim_state': opt.state_dict(),
}
with tempfile.NamedTemporaryFile(suffix='.pt', delete=False) as f:
    ckpt_path = f.name
torch.save(ckpt, ckpt_path)
print(f'저장 완료: {ckpt_path}')

# 복원: 같은 구조의 모델/옵티마이저를 먼저 만들고 state 만 부어 넣는다
torch.manual_seed(0)   # 일부러 다른 초기화
model2 = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 2))
opt2 = optim.Adam(model2.parameters(), lr=1e-3)

loaded = torch.load(ckpt_path, weights_only=False)
model2.load_state_dict(loaded['model_state'])
opt2.load_state_dict(loaded['optim_state'])
print(f'복원 완료: epoch={loaded["epoch"]}, best_loss={loaded["best_loss"]}')

# 두 모델의 첫 가중치가 같은지 확인
w1 = next(model.parameters()).detach().flatten()[:4]
w2 = next(model2.parameters()).detach().flatten()[:4]
print('원본 weight[:4] :', w1.numpy().round(4))
print('복원 weight[:4] :', w2.numpy().round(4))
print('일치 여부      :', torch.allclose(w1, w2))

# 9. 흔한 함정 (Common Pitfalls)

학습 코드에서 자주 부딪히는 5가지:

1. **`optimizer.zero_grad()` 누락** → grad 누적, 학습 폭주
2. **`.item()` 누락** → loss 텐서 누적이 그래프 메모리를 잡고 있어 OOM
3. **`model.eval()` 누락** → Dropout/BN 이 train 모드로 동작해 추론 결과가 매번 다르거나 통계가 망가짐
4. **모델만 `.to(device)`, 데이터 이동 누락** → device mismatch 에러
5. **`requires_grad=False` 인 파라미터** → 의도치 않게 freeze 되어 학습 안 됨 (예: backbone freeze 가 head 까지 잠금)

In [ ]:
# 함정 1: zero_grad 누락 — grad norm 이 폭주
torch.manual_seed(SEED)
m = nn.Linear(4, 1)
opt = optim.SGD(m.parameters(), lr=0.0)   # lr=0 으로 step 영향 제거
xb, yb = torch.randn(8, 4), torch.randn(8, 1)
loss_fn = nn.MSELoss()
for step in range(3):
    # opt.zero_grad()  # <- 빠뜨림
    loss_fn(m(xb), yb).backward()
    print(f'[함정 1] step {step}: weight.grad norm = {m.weight.grad.norm().item():.4f}')

In [ ]:
# 함정 2: .item() 누락 — 텐서를 그래프째로 누적
torch.manual_seed(SEED)
m = nn.Linear(50, 50)
opt = optim.SGD(m.parameters(), lr=1e-3)
xb, yb = torch.randn(16, 50), torch.randn(16, 50)
loss_fn = nn.MSELoss()

bad, good = [], []
for _ in range(3):
    opt.zero_grad()
    loss = loss_fn(m(xb), yb)
    loss.backward()
    opt.step()
    bad.append(loss)         # 그래프가 살아 있다
    good.append(loss.item()) # float 만 보관

print('[함정 2] bad[0].grad_fn  =', bad[0].grad_fn, '<- 그래프가 메모리를 잡고 있음')
print('[함정 2] good[0]         =', good[0], '<- 단순 float 라 안전')

In [ ]:
# 함정 3: eval() 누락 — Dropout 때문에 추론 결과가 매번 달라진다
torch.manual_seed(SEED)
m = nn.Sequential(nn.Linear(8, 8), nn.Dropout(0.5), nn.Linear(8, 1))
x = torch.ones(1, 8)

# 일부러 train 모드로 추론
m.train()
preds = [m(x).item() for _ in range(3)]
print('[함정 3] train 모드 추론:', [round(p, 4) for p in preds], '<- 매번 다름')

m.eval()
preds = [m(x).item() for _ in range(3)]
print('[함정 3] eval  모드 추론:', [round(p, 4) for p in preds], '<- 항상 같음')

In [ ]:
# 함정 4: 모델만 device 이동, 데이터 이동 누락
device = get_device()
m = nn.Linear(4, 1).to(device)
xb_cpu = torch.randn(3, 4)   # CPU 그대로

if device.type != 'cpu':
    try:
        _ = m(xb_cpu)
    except RuntimeError as e:
        print('[함정 4] device mismatch:', str(e).splitlines()[0])
    print('[함정 4] 해결: xb = xb.to(device) 한 줄')
else:
    print('[함정 4] 현재 device=cpu — 모델/데이터 모두 cpu 라 에러 없음.')
    print('         GPU 환경에서는 model 만 .to(device) 후 데이터를 안 옮기면 RuntimeError.')

In [ ]:
# 함정 5: requires_grad=False 인 파라미터를 모르고 학습 안 됨
torch.manual_seed(SEED)
backbone = nn.Linear(8, 8)
head = nn.Linear(8, 1)

# 의도: backbone 만 freeze 하고 head 만 학습
for p in backbone.parameters():
    p.requires_grad = False

# 잘못된 패턴: 두 모듈을 합친 후 model.parameters() 전부를 optimizer 에 줌
model = nn.Sequential(backbone, head)
opt = optim.SGD(model.parameters(), lr=1e-2)   # backbone 의 frozen param 도 들어감
print('[함정 5] optimizer 가 받은 파라미터 중 trainable 비율:')
trainable = sum(p.requires_grad for p in model.parameters())
total = sum(1 for _ in model.parameters())
print(f'         {trainable}/{total} (frozen 포함되어 있어도 grad 가 None 이라 step 무시)')

# 더 명확한 패턴: trainable 만 골라서 옵티마이저에 넘긴다
opt2 = optim.SGD([p for p in model.parameters() if p.requires_grad], lr=1e-2)
print('[함정 5] 권장 패턴: filter(requires_grad) 후 옵티마이저에 전달')
print(f'         opt2 가 받은 파라미터 수 = {sum(1 for _ in opt2.param_groups[0]["params"])}')

# 정리

| 부품 | 책임 |
|------|------|
| `nn.Module` | parameter (학습됨) + buffer (저장만) 관리, forward 정의 |
| `Optimizer` | `param_groups` 별 갱신 규칙, `step()` 과 `zero_grad()` |
| `Dataset` / `DataLoader` | 단일 샘플 로직 / 배치화·셔플·병렬로딩 |
| `train()` / `eval()` | Dropout/BN 의 동작 모드 전환 |
| `torch.no_grad()` | 그래프 추적 OFF (메모리/속도) |
| `.to(device)` | 모델·텐서를 같은 device 로 통일 |
| `autocast` + `GradScaler` | 혼합정밀도 학습 (GPU) |
| `state_dict` 저장 | 클래스 정의 의존성 없는 안전한 체크포인트 |

01 노트북의 autograd + 이 노트북의 학습 부품을 합치면, "학습 한 step" 의 모든 부분이
**왜 그렇게 생겼는지** 가 설명된다. 다음 학습 모듈에서는 이 위에 CNN/RNN/Transformer 를 얹는다.